# Faster R-CNN


In [14]:
import subprocess, sys, warnings
warnings.filterwarnings("ignore")
for pkg in ["pycocotools", "matplotlib"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU T4 x2")
print("CUDA OK", torch.__version__)

CUDA OK 2.10.0+cu128


In [15]:
import json, os, csv, random
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import (fasterrcnn_resnet50_fpn_v2,FasterRCNN_ResNet50_FPN_V2_Weights,)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


KAGGLE_INPUT_ROOT = "/kaggle/input/datasets/nikachuu/data-prep"
WORKING_CONFIG = "/kaggle/working/config.json"
INPUT_CONFIG = os.path.join(KAGGLE_INPUT_ROOT, "config.json")

def load_config():
    if os.path.isfile(WORKING_CONFIG):
        config_path = WORKING_CONFIG
    elif os.path.isfile(INPUT_CONFIG):
        config_path = INPUT_CONFIG
    else:
        raise FileNotFoundError
    with open(config_path) as f:
        config = json.load(f)
    roots = [
        config.get("dataset_path"),
        os.path.join(KAGGLE_INPUT_ROOT, "dataset"),
        "/kaggle/working/dataset",]
    for root in roots:
        if not root:
            continue
        train_ann = os.path.join(root, "train", "_annotations.coco.json")
        if os.path.isfile(train_ann):
            config["dataset_path"] = root
            config["train_ann"] = train_ann
            config["valid_ann"] = os.path.join(root, "valid", "_annotations.coco.json")
            config["test_ann"] = os.path.join(root, "test", "_annotations.coco.json")
            config["train_img_dir"] = os.path.join(root, "train")
            config["valid_img_dir"] = os.path.join(root, "valid")
            config["test_img_dir"] = os.path.join(root, "test")
            break
    else:
        raise FileNotFoundError
    config["save_dir"] = config.get("save_dir") or "/kaggle/working/results"
    os.makedirs(config["save_dir"], exist_ok=True)
    return config, config_path

config, CONFIG_PATH = load_config()

Experiment_Name = config["Experiment_Name"]
dataset_version = config["dataset_version"]
Confidence_Threshold = config["Confidence_Threshold"]
epochs_rcnn = config["epochs_rcnn"]
lr_rcnn = config["lr_rcnn"]
PATIENCE = config["PATIENCE"]
SEED = config["SEED"]
save_dir = config["save_dir"]
train_ann = config["train_ann"]
valid_ann = config["valid_ann"]
test_ann = config["test_ann"]
train_img_dir = config["train_img_dir"]
valid_img_dir = config["valid_img_dir"]
test_img_dir = config["test_img_dir"]
log_file = os.path.join(save_dir, "results.csv")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def log_result(exp, model, acc, prec, rec, f1, thresh, variant, ds_ver, notes=""):
    row = {"timestamp": datetime.now().isoformat(timespec="seconds"),"experiment": exp, "model": model,"accuracy": f"{acc:.2f}", "precision": f"{prec:.2f}","recall": f"{rec:.2f}", "f1": f"{f1:.2f}","threshold": thresh, "variant": variant,"dataset_version": ds_ver, "notes": notes,}
    write_header = not os.path.exists(log_file)
    with open(log_file, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            w.writeheader()
        w.writerow(row)
    print("Logged to", log_file)

print("Device:", DEVICE)

Device: cuda


In [16]:
class RhizomeDetectionDataset(Dataset):
    def __init__(self, ann_file, img_dir, augment=False):
        with open(ann_file) as f:
            coco = json.load(f)
        self.img_dir = img_dir
        self.images  = coco["images"]
        self.ann_map = {}
        for ann in coco["annotations"]:
            self.ann_map.setdefault(ann["image_id"], []).append(ann)
        self.aug = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.10, contrast=0.10),
        ]) if augment else None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        info = self.images[idx]
        img  = Image.open(os.path.join(self.img_dir, info["file_name"])).convert("RGB")
        if self.aug:
            img = self.aug(img)
        boxes, labels = [], []
        for ann in self.ann_map.get(info["id"], []):
            x, y, w, h = [float(v) for v in ann["bbox"]]
            if w > 2 and h > 2:
                W, H  = img.size
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(W, x + w), min(H, y + h)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(ann["category_id"])
        boxes  = torch.tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0, 4))
        labels = torch.tensor(labels, dtype=torch.int64)   if labels else torch.zeros((0,), dtype=torch.int64)
        return transforms.ToTensor()(img), {
            "boxes":    boxes,
            "labels":   labels,
            "image_id": torch.tensor([info["id"]]),}

def collate_fn(batch):
    return tuple(zip(*batch))


train_det = RhizomeDetectionDataset(train_ann, train_img_dir, augment=True)
valid_det = RhizomeDetectionDataset(valid_ann, valid_img_dir, augment=False)
test_det  = RhizomeDetectionDataset(test_ann,  test_img_dir,  augment=False)

train_det_loader = DataLoader(train_det, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=0)
valid_det_loader = DataLoader(valid_det, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_det_loader  = DataLoader(test_det,  batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

with open(train_ann) as f:
    info = json.load(f)
NUM_CLASSES = len(info["categories"]) + 1
print("Num classes:", NUM_CLASSES)

rcnn_model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,trainable_backbone_layers=3,).to(DEVICE)
in_feat = rcnn_model.roi_heads.box_predictor.cls_score.in_features
rcnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NUM_CLASSES)
print("Model Initiated")

Num classes: 5
Model Initiated


In [17]:
class EarlyStopping:
    def __init__(self, patience=5, mode="min"):
        self.patience, self.mode = patience, mode
        self.counter, self.best, self.stop = 0, None, False
    def __call__(self, val):
        if self.best is None:
            self.best = val
            return
        improved = val < self.best - 0.001 if self.mode == "min" else val > self.best + 0.001
        if improved:
            self.best, self.counter = val, 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

class FasterRCNNTrainer:
    def __init__(self, model, device, lr=0.005, patience=5):
        self.model = model
        self.device = device
        self.save_path = os.path.join(save_dir, "rcnn_best.pth")
        self.history = {"train_loss": [], "val_loss": []}
        self.best_val = float("inf")
        self.optimizer = optim.SGD(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr, momentum=0.9, weight_decay=0.0005)
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=epochs_rcnn, eta_min=1e-6)
        self.es = EarlyStopping(patience=patience)

    def _epoch(self, loader, train=True):
        self.model.train() if train else self.model.eval()
        total, n = 0.0, 0
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for imgs, tgts in loader:
                imgs = [i.to(self.device) for i in imgs]
                tgts = [{k: v.to(self.device) for k, v in t.items()} for t in tgts]
                loss = sum(self.model(imgs, tgts).values())
                if train:
                    self.optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.optimizer.step()
                total += loss.item()
                n += 1
        return total / n if n else 0.0

    def fit(self, train_loader, valid_loader, epochs):
        for epoch in range(epochs):
            tr = self._epoch(train_loader, True)
            vl = self._epoch(valid_loader, False)
            self.scheduler.step()
            self.history["train_loss"].append(tr)
            self.history["val_loss"].append(vl)
            print("Epoch", epoch + 1, "train", round(tr, 4), "val", round(vl, 4))
            if vl < self.best_val:
                self.best_val = vl
                torch.save(self.model.state_dict(), self.save_path)
                print("Saved best val", round(vl, 4))
            self.es(vl)
            if self.es.stop:
                print("Early stop epoch", epoch + 1)
                break

    def evaluate(self, test_loader, conf_thresh=0.5, iou_thresh=0.5):
        self.model.load_state_dict(torch.load(self.save_path, map_location=self.device))
        self.model.eval()
        tp = fp = fn = 0
        iou_list = []
        def iou_fn(b1, b2):
            x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
            x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
            inter = max(0, x2 - x1) * max(0, y2 - y1)
            union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
            return inter / union if union > 0 else 0
        with torch.no_grad():
            for imgs, tgts in test_loader:
                imgs = [i.to(self.device) for i in imgs]
                preds = self.model(imgs)
                for pred, tgt in zip(preds, tgts):
                    gt = tgt["boxes"].cpu().numpy()
                    pb = pred["boxes"].cpu().numpy()
                    ps = pred["scores"].cpu().numpy()
                    pb = pb[ps >= conf_thresh]
                    matched = set()
                    for box in pb:
                        best_i, best_j = 0, -1
                        for j, gb in enumerate(gt):
                            if j in matched:
                                continue
                            v = iou_fn(box, gb)
                            if v > best_i:
                                best_i, best_j = v, j
                        if best_i >= iou_thresh:
                            tp += 1
                            matched.add(best_j)
                            iou_list.append(best_i)
                        else:
                            fp += 1
                    fn += len(gt) - len(matched)
        prec = tp / (tp + fp) if (tp + fp) else 0
        rec = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
        miou = float(np.mean(iou_list)) if iou_list else 0
        print("Precision", round(prec * 100, 2), "Recall", round(rec * 100, 2),
              "F1", round(f1 * 100, 2), "mIoU", round(miou * 100, 2))
        return prec, rec, f1, miou

    def plot(self):
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].plot(self.history["train_loss"], label="train")
        ax[0].plot(self.history["val_loss"], label="val")
        ax[0].legend()
        ax[0].set_title("R-CNN loss")
        gap = [v - t for v, t in zip(self.history["val_loss"], self.history["train_loss"])]
        ax[1].bar(range(len(gap)), gap)
        ax[1].set_title("Val minus train")
        plt.savefig(os.path.join(save_dir, "rcnn_" + Experiment_Name + ".png"), dpi=150)
        plt.show()

rcnn_trainer = FasterRCNNTrainer(rcnn_model, DEVICE, lr=lr_rcnn, patience=PATIENCE)
rcnn_trainer.fit(train_det_loader, valid_det_loader, epochs_rcnn)
rcnn_trainer.plot()
rp, rr, rf, riou = rcnn_trainer.evaluate(test_det_loader, Confidence_Threshold, 0.5)
config["rcnn_save_path"] = rcnn_trainer.save_path
with open("/kaggle/working/config.json", "w") as f:
    json.dump(config, f, indent=2)
log_result(Experiment_Name, "FasterRCNN-R50-FPN-V2", rp*100, rp*100, rr*100, rf*100,
           Confidence_Threshold, "ResNet50-FPN-V2", dataset_version,
           "mIoU=" + str(round(riou*100, 2)))
print("Done")

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)